#  Guia de Exploração do Laboratório Virtual

Bem-vindo à simulação interativa da **Hodógrafa de Hamilton**. Esta ferramenta foi desenhada para ajudá-lo a visualizar a conexão geométrica profunda entre a forma de uma órbita no espaço (onde o planeta está) e o seu comportamento no espaço de velocidades (o quão rápido ele está indo).

## Como interpretar os painéis?

*   **Painel Esquerdo (Espaço de Posições):** Mostra a órbita física do corpo celeste. O ponto laranja representa o corpo central atrator (como o Sol) localizado em um dos focos da cônica.
*   **Painel Direito (Espaço de Velocidades - Hodógrafa):** Aqui está o "segredo" de Hamilton. Independentemente da forma da órbita à esquerda, a trajetória do vetor velocidade é **sempre uma circunferência** de raio $R_h$.
    *   A bolinha preta é a **Origem $(v=0)$**.
    *   O "x" vermelho é o **Centro da Hodógrafa**, que está deslocado da origem por uma distância $d$.
    *   As setas representam os vetores de velocidade no periastro ($\vec{v}_p$, mais rápido) e no apoastro ($\vec{v}_a$, mais lento).

---

## Situações interessantes para você investigar:

Utilize os controles deslizantes para alterar as velocidades $v_1$ e $v_2$. O código automaticamente definirá a maior como o periastro ($v_p$) e a menor como o apoastro ($v_a$). Teste os seguintes cenários:

### 1. A Órbita Circular (O caso perfeito)
*   **Ação:** Ajuste os dois controles para o mesmo valor (ex: $v_1 = 1.5$ e $v_2 = 1.5$).
*   **O que observar:** A órbita se torna um círculo perfeito. Na hodógrafa, o deslocamento $d$ se torna zero e a Origem $(0,0)$ passa a coincidir exatamente com o centro do círculo. A excentricidade é $e = 0$.

### 2. A Órbita Elíptica (O movimento planetário padrão)
*   **Ação:** Deixe $v_1$ maior que $v_2$ (ex: $v_1 = 2.0$ e $v_2 = 0.8$).
*   **O que observar:** O centro da hodógrafa se desloca verticalmente. Note que **a origem $(0,0)$ permanece dentro do círculo vermelho**. Sempre que a origem estiver no interior da hodógrafa, a órbita correspondente será uma elipse (fechada) com $0 < e < 1$.

### 3. O Ponto de Escape (A Órbita Parabólica)
*   **Ação:** Mantenha $v_1 = 2.0$ e arraste $v_2$ lentamente até **exatamente $0.0$**.
*   **O que observar:**  À medida que a velocidade no apoastro zera, o círculo da hodógrafa sobe até que **a borda toque exatamente a origem $(0,0)$**. O raio $R_h$ torna-se igual ao deslocamento $d$. Isso força a excentricidade a ser $e = 1$, rompendo a elipse no painel esquerdo e transformando-a em uma trajetória parabólica aberta.

### 4. O Significado das Setas (O Raio e o Deslocamento)
*   **Ação:** Escolha quaisquer valores arbitrários para $v_1$ e $v_2$.
*   **O que observar:** Olhe para o painel direito. Observe como o raio da hodógrafa ($R_h$) é perfeitamente descrito pela média do tamanho das duas setas: $(v_p + v_a)/2$. Já o deslocamento do centro ($d$) é a semidiferença: $(v_p - v_a)/2$. A geometria reflete diretamente a álgebra.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets

# Garante que os gráficos apareçam logo abaixo da célula no Colab
%matplotlib inline

# Configuração Tipográfica (Estilo Artigo Acadêmico)
plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "axes.formatter.use_mathtext": True,
    "font.size": 11,
})

def simular_hodografa(v_1, v_2):
    # Trava Física: Garante que v_p (periastro) >= v_a (apoastro)
    v_p = max(v_1, v_2)
    v_a = min(v_1, v_2)

    # 1. NÚCLEO FÍSICO
    R_h = (v_p + v_a) / 2.0
    d = (v_p - v_a) / 2.0
    e = d / R_h if R_h != 0 else 0.0

    # 2. ESPAÇO DE POSIÇÕES (Órbita)
    theta = np.linspace(0, 2 * np.pi, 1000)
    p_semilatus = 1.0

    if abs(e) < 1:
        # Elíptica / Circular
        r = p_semilatus / (1 + e * np.cos(theta))
        x_orbit, y_orbit = r * np.cos(theta), r * np.sin(theta)
        orbit_label = f"Órbita Elíptica ($e = {e:.2f}$)"
    elif abs(e - 1) < 1e-6:
        # Parabólica
        theta_p = np.linspace(-2.5, 2.5, 1000)
        r = p_semilatus / (1 + e * np.cos(theta_p))
        x_orbit, y_orbit = r * np.cos(theta_p), r * np.sin(theta_p)
        orbit_label = f"Órbita Parabólica ($e = {e:.2f}$)"
    else:
        # Hiperbólica
        theta_max = np.arccos(-1 / e) - 0.05
        theta_h = np.linspace(-theta_max, theta_max, 1000)
        r = p_semilatus / (1 + e * np.cos(theta_h))
        x_orbit, y_orbit = r * np.cos(theta_h), r * np.sin(theta_h)
        orbit_label = f"Órbita Hiperbólica ($e = {e:.2f}$)"

    # 3. ESPAÇO DE VELOCIDADES (Hodógrafa)
    phi = np.linspace(0, 2 * np.pi, 500)
    vx_circle = R_h * np.cos(phi)
    vy_circle = R_h * np.sin(phi) + d

    # 4. VISUALIZAÇÃO (Painel Duplo)
    fig, (ax_pos, ax_vel) = plt.subplots(1, 2, figsize=(12, 5.5))

    # --- Painel Esquerdo: Espaço de Posições ---
    ax_pos.plot(x_orbit, y_orbit, color='royalblue', lw=2.5, label=orbit_label)
    ax_pos.scatter([0], [0], color='orange', s=150, zorder=5, edgecolor='black', label='Sol (Foco)')

    ax_pos.axhline(0, color='gray', lw=1, ls='--', alpha=0.5)
    ax_pos.axvline(0, color='gray', lw=1, ls='--', alpha=0.5)
    ax_pos.set_title("Espaço de Posições", weight='bold')
    ax_pos.set_xlabel("$x$")
    ax_pos.set_ylabel("$y$")
    ax_pos.set_aspect('equal')
    ax_pos.set_xlim(-3, 3) # Limite fixo para a câmera não tremer
    ax_pos.set_ylim(-3, 3)
    ax_pos.legend(loc='upper right')
    ax_pos.grid(alpha=0.2)

    # --- Painel Direito: Espaço de Velocidades (Hodógrafa) ---
    ax_vel.plot(vx_circle, vy_circle, color='crimson', lw=2.5, label=f'Hodógrafa ($R_h = {R_h:.2f}$)')

    ax_vel.scatter([0], [0], color='black', s=80, zorder=6, label='Origem $(v=0)$')
    ax_vel.scatter([0], [d], color='crimson', s=80, marker='x', zorder=6, label=f'Centro ($d = {d:.2f}$)')

    # Vetores de Velocidade
    ax_vel.annotate('', xy=(0, v_p), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
    ax_vel.text(0.1, v_p/2, r'$\vec{v}_p$', color='blue', fontsize=14, weight='bold')

    if v_a > 0:
        ax_vel.annotate('', xy=(0, -v_a), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
        ax_vel.text(0.1, -v_a/2, r'$\vec{v}_a$', color='green', fontsize=14, weight='bold')

    ax_vel.axhline(0, color='gray', lw=1, ls='--', alpha=0.5)
    ax_vel.axvline(0, color='gray', lw=1, ls='--', alpha=0.5)
    ax_vel.set_title("Espaço de Velocidades (Hodógrafa)", weight='bold')
    ax_vel.set_xlabel("$v_x$")
    ax_vel.set_ylabel("$v_y$")
    ax_vel.set_aspect('equal')
    ax_vel.set_xlim(-3.5, 3.5)
    ax_vel.set_ylim(-3.5, 3.5)
    ax_vel.legend(loc='upper right')
    ax_vel.grid(alpha=0.2)

    # Título Principal com Parâmetros
    fig.suptitle(f"Parâmetros Físicos: $v_p = {v_p:.2f}$  |  $v_a = {v_a:.2f}$   ||   $R_h = {R_h:.2f}$  |  $d = {d:.2f}$  |  $e = {e:.2f}$",
                 fontsize=13, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round,pad=0.5'))

    plt.tight_layout()
    plt.subplots_adjust(top=0.88)
    plt.show()

# ==========================================================================
# INTERFACE INTERATIVA (Colab Nativo)
# ==========================================================================
interact(simular_hodografa,
         v_1=widgets.FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1, description='Velocidade 1:'),
         v_2=widgets.FloatSlider(value=0.5, min=0.0, max=3.0, step=0.1, description='Velocidade 2:'));

interactive(children=(FloatSlider(value=1.5, description='Velocidade 1:', max=3.0), FloatSlider(value=0.5, des…

---

## 🔭 Conectando a Simulação com as Leis de Kepler

A verdadeira beleza da hodógrafa de Hamilton é que ela carrega em sua geometria a essência das Três Leis de Kepler. Utilize a simulação acima para observar como essas leis clássicas se manifestam:

### 1ª Lei de Kepler (Lei das Órbitas)
> *"Os planetas descrevem órbitas elípticas, com o Sol ocupando um dos focos."*
*   **Na Simulação:** Olhe para o painel da esquerda (Espaço de Posições). A cônica gerada é sempre uma elipse (ou parábola/hipérbole) com o atrator (Sol) firmemente ancorado na origem (Foco).
*   **O Segredo da Hodógrafa:** Por que a órbita é uma cônica perfeita? A hodógrafa nos mostra que, sob uma força inversamente proporcional ao quadrado da distância (gravidade), o vetor velocidade sempre descreve um círculo. É a excentricidade geométrica desse círculo de velocidades ($e = d/R_h$) que obriga a trajetória real a assumir a forma de uma elipse!

### 2ª Lei de Kepler (Lei das Áreas)
> *"O segmento de reta que une o Sol a um planeta varre áreas iguais em intervalos de tempo iguais."*
*   **Na Simulação:** Observe o tamanho dos vetores no painel da direita (Hodógrafa). O vetor azul ($\vec{v}_p$, no periastro) é sempre muito maior que o vetor verde ($\vec{v}_a$, no apoastro).
*   **O Segredo da Hodógrafa:** Para que a Lei das Áreas seja verdadeira, o momento angular ($L = mvr\sin\theta$) tem que ser conservado. A simulação mostra visualmente que, quando o planeta está mais perto do Sol, ele **precisa** estar mais rápido (vetor azul grande) para compensar a perda de distância. Na hodógrafa, o vetor velocidade alcança o seu máximo esticando-se até o topo do círculo.

### 3ª Lei de Kepler (Lei dos Períodos)
> *"O quadrado do período de revolução de um planeta é proporcional ao cubo do semieixo maior de sua órbita."*
*   **O Segredo da Hodógrafa:** Interagindo cos controles e aumente ambas as velocidades ($v_1$ e $v_2$), mantendo-as próximas (baixa excentricidade). Você notará que o raio da hodógrafa ($R_h$) aumenta. Fisicamente, se um planeta possui velocidades globais maiores, ele deve estar em uma órbita mais interna (menor semieixo maior), completando seu "ano" orbital muito mais rápido. A hodógrafa nos lembra que planetas velozes moram em hodógrafas grandes e órbitas pequenas.